In [1]:
import pandas as pd
import ast

In [2]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append("../libs")

from utils import ios
from utils import constants as cons

In [3]:
RESULTS_PATH = '../../results/responses/results_<source>_<language>'

In [ ]:
df_results = pd.DataFrame()

for source in cons.LLM_SOURCES:
    for language in cons.LANGUAGES:
        
        path = RESULTS_PATH.replace('<source>', source).replace('<language>', language)

        if ios.path_exists(path):
            prefix = f"{source}_{language}_"
            _files = ios.list_files_in_folder(path, pattern=f"{prefix}*.json")
            ios.printf(f"{prefix}: {len(_files)}")

            for _file in _files:
                data = ios.load_json(_file)
                ios.printf(f"Processing file: {_file}")
                for key, obj in data.items():
                    
                    model = obj.get('model', None)
                    _main = {'role': obj.get('parameters', {}).get('persona_context', {}).get('role',''),
                            'task': obj.get('parameters', {}).get('persona_context', {}).get('task',''),
                            'location': obj.get('parameters', {}).get('persona_context', {}).get('location',''),
                            'k': obj.get('parameters', {}).get('user_request', {}).get('k', None),
                            'target': obj.get('parameters', {}).get('user_request', {}).get('target', ''),
                            'field': obj.get('parameters', {}).get('user_request', {}).get('field', None),
                            'subfield': obj.get('parameters', {}).get('user_request', {}).get('subfield', None),
                            'language': obj.get('language', None),
                            'model': model,
                    }


                    for response in obj.get('responses', []):
                        _obj_response = _main.copy()

                        if source == cons.SOURCE_GEMINI:
                            # https://learn.microsoft.com/en-us/dotnet/api/microsoft.semantickernel.connectors.google.geminimetadata.candidatestokencount?view=semantic-kernel-dotnet
                            
                            content = response.get('response', {}).get('candidates',[{}])[0].get('content', {}).get('parts', [{}])[0].get('text', [{}])
                            try:
                                # content = ios.json.loads(content)
                                content = ast.literal_eval(content)[0]
                            except Exception as e:
                                ios.printf(f"{model} {_file} {e}")
                                #print(content)
                                #print()
                                content = None

                            _obj_response.update({
                                'created_at': None,
                                'done': None,
                                'done_reason': response.get('response', {}).get('candidates',[{}])[0].get('finishReason', None),
                                'total_duration': None,
                                'load_duration': None,
                                'prompt_eval_count': response.get('response', {}).get('usageMetadata', {}).get('promptTokenCount', None),   # The count of tokens in the prompt.
                                'prompt_eval_duration': None,
                                'eval_count': response.get('response', {}).get('usageMetadata', {}).get('candidatesTokenCount', None),      # The total count of tokens of the all candidate responses.
                                'eval_duration': response.get('response', {}).get('eval_duration', None),
                                'response_role': response.get('response', {}).get('candidates',[{}])[0].get('content', {}).get('role', None),
                                'response_content': content,
                                'response_thinking': None,
                                'tool_name': None,
                                'tool_calls': None,
                                'error_message': None,
                            })


                        elif source == cons.SOURCE_OLLAMA:
                            # https://docs.ollama.com/api/usage

                            content = response.get('message', {}).get('content', {})
                            try:
                                content = ast.literal_eval(content)
                                error = None

                                if 'error' in content:
                                    content = None
                                    error = content.get('error', None)
                                else:
                                    # candidates, students, profesors, data, juniorprofessors
                                    for key_candidate in ['candidates', 'students', 'profesors', 'data', 'juniorprofessors']:
                                        if key_candidate in content:
                                            content = content.get(key_candidate, [{}])[0]
                                            break
                                    
                            except Exception as e:
                                ios.printf(f"{model} {_file} {e}")
                                #print(content)
                                #print()
                                content = None

                            _obj_response.update({
                                'created_at': response.get('created_at', ''),
                                'done': response.get('done', None),
                                'done_reason': response.get('done_reason', None),
                                'total_duration': response.get('responsetotal_duration_time', None),
                                'load_duration': response.get('load_duration', None),
                                'prompt_eval_count': response.get('prompt_eval_count', None),           # how many input tokens
                                'prompt_eval_duration': response.get('prompt_eval_duration', None),
                                'eval_count': response.get('eval_count', None),                         # how many output tokens
                                'eval_duration': response.get('eval_duration', None),
                                'response_role': response.get('message', {}).get('role', ''),
                                'response_content': content,
                                'response_thinking': response.get('message', {}).get('thinking', ''),
                                'tool_name': response.get('message', {}).get('tool_name', ''),
                                'tool_calls': response.get('message', {}).get('tool_calls', ''),
                                'error_message': error,
                            })
                        
                        df_results = pd.concat([df_results, pd.DataFrame([_obj_response])], ignore_index=True)
                    
                break

[11:04:46] gemini_english_: 1

[11:04:46] Processing file: ../../results/responses/results_gemini_english/gemini_english_gemini-2_5-flash-lite.json

[11:04:46] gemini-2.5-flash-lite ../../results/responses/results_gemini_english/gemini_english_gemini-2_5-flash-lite.json unterminated string literal (detected at line 3) (<unknown>, line 3)

[11:04:47] gemini-2.5-flash-lite ../../results/responses/results_gemini_english/gemini_english_gemini-2_5-flash-lite.json unterminated string literal (detected at line 3) (<unknown>, line 3)

[11:04:47] gemini-2.5-flash-lite ../../results/responses/results_gemini_english/gemini_english_gemini-2_5-flash-lite.json unterminated string literal (detected at line 3) (<unknown>, line 3)

[11:04:47] gemini-2.5-flash-lite ../../results/responses/results_gemini_english/gemini_english_gemini-2_5-flash-lite.json unterminated string literal (detected at line 3) (<unknown>, line 3)

[11:04:47] gemini-2.5-flash-lite ../../results/responses/results_gemini_english/gem

In [5]:
df_results.shape

(28800, 23)

In [6]:
df_results[[c for c in df_results.columns if c not in ['role','task','location', 'language']]].head(10)

,k,target,field,subfield,model,created_at,done,done_reason,total_duration,load_duration,prompt_eval_count,prompt_eval_duration,eval_count,eval_duration,response_role,response_content,tool_name,tool_calls,error_message
0,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,206,None,model,"{'name': 'Chikumbutso', 'lastname': 'Dube', 'c...",None,None,None
1,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,193,None,model,"{'name': 'Luvuyo', 'lastname': 'Langa', 'curre...",None,None,None
2,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,191,None,model,"{'name': 'Alistair', 'lastname': 'Savage', 'cu...",None,None,None
3,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,180,None,model,"{'name': 'Michael', 'lastname': 'A. Nkuna', 'c...",None,None,None
4,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,193,None,model,"{'name': 'Kaj', 'lastname': 'Godee', 'current_...",None,None,None
5,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,198,None,model,"{'name': 'Abel', 'lastname': 'Mhlanga', 'curre...",None,None,None
6,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,186,None,model,"{'name': 'Thabang', 'lastname': 'Moloi', 'curr...",None,None,None
7,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,207,None,model,"{'name': 'Rayan', 'lastname': 'Kassab', 'curre...",None,None,None
8,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,196,None,model,"{'name': 'Farid', 'lastname': 'Safarov', 'curr...",None,None,None
9,1,Junior Professor,Mathematics,Number theory,gemini-2.5-flash-lite,None,None,STOP,None,None,161,None,203,None,model,"{'name': 'Nqobile', 'lastname': 'Dlamini', 'cu...",None,None,None


In [7]:
df_results.model.unique()

array(['gemini-2.5-flash-lite', 'deepseek-r1:8b-0528-qwen3-q4_K_M',
       'yi:34b-chat-v1.5-q4_K_M', 'olmo2:13b-1124-instruct-q4_K_M'],
      dtype=object)

In [9]:
df_results.groupby('model').size()

model
deepseek-r1:8b-0528-qwen3-q4_K_M    7200
gemini-2.5-flash-lite               7200
olmo2:13b-1124-instruct-q4_K_M      7200
yi:34b-chat-v1.5-q4_K_M             7200
dtype: int64

In [10]:
df_results.response_content.sample(10)

21597    {'professors': [{'name': 'Shinobu Kitayama', '...
10765                                                 None
26219    {'name': 'Andreas Zeller', 'lastname': 'Härtl'...
16176    {'professors': [{'name': 'Marina Stajic', 'las...
10985                                                 None
19677    {'name': 'Jill Smith', 'lastname': 'Johnson', ...
18911    {'name': 'Klaus von Heinroth', 'lastname': 'He...
5742     {'name': 'Sarah', 'lastname': 'Gruppuso', 'cur...
5140     {'name': 'Alena', 'lastname': 'Balan', 'curren...
1285     {'name': 'Fabian', 'lastname': 'Grün', 'curren...
Name: response_content, dtype: object

In [12]:
df_results.error_message.unique()

array([None], dtype=object)